# LIAR DATASET

In [ ]:
import kagglehub
import os
import shutil
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

from hypothesaes.quickstart import train_sae, interpret_sae, generate_hypotheses, evaluate_hypotheses
from hypothesaes.embedding import get_openai_embeddings, get_local_embeddings

INTERPRETER_MODEL = "gpt-4.1"
ANNOTATOR_MODEL = "gpt-4.0-mini"
N_WORKERS_ANNOTATION = 8

In [18]:
# source_path = kagglehub.dataset_download("doanquanvietnamca/liar-dataset")
# current_dir = os.getcwd()
# destination_path = os.path.join(current_dir, "liar_data")
# shutil.move(source_path, destination_path)

In [19]:
column_names = ['id', 'label', 'statement', 'subject', 'speaker', 'title', 'state', 'party', 'barely_true', 'false', 'half_true', 'mostly_true', 'pants_on_fire', 'context']
def clean_label(lbl):
    if lbl in ['false', 'pants-fire']:
        return 0
    elif lbl in ['true', 'mostly-true']:
        return 1

def preprocess_liar_data(path):
    df = pd.read_csv(path, sep="\t", names=column_names)
    df["binary_label"] = df['label'].apply(clean_label)
    cleaned_df = df[["id", "statement", "binary_label"]].dropna()
    cleaned_df['binary_label'] = cleaned_df['binary_label'].astype(int)
    return cleaned_df

In [20]:
train_df = preprocess_liar_data('liar_data/train.tsv')
val_df = preprocess_liar_data('liar_data/valid.tsv')
test_df = preprocess_liar_data('liar_data/test.tsv')

In [21]:
texts = train_df['statement'].tolist()
labels = train_df['binary_label'].values

val_texts = val_df['statement'].tolist()
val_labels = val_df['binary_label'].values

test_texts = test_df['statement'].tolist()
test_labels = test_df['binary_label'].values

In [22]:
# EMBEDDER = "text-embedding-3-small" # OpenAI
EMBEDDER = "nomic-ai/modernbert-embed-base" # HuggingFace
# EMBEDDER = "mixedbread-ai/mxbai-embed-large-v1" # HuggingFace

CACHE_NAME = f"liar_quickstart_{EMBEDDER}"

# text2embedding = get_openai_embeddings(texts + val_texts + test_texts, model=EMBEDDER, cache_name=CACHE_NAME)
text2embedding = get_local_embeddings(texts + val_texts + test_texts, model=EMBEDDER, cache_name=CACHE_NAME)

train_embeddings = np.stack([text2embedding[text] for text in texts])
val_embeddings = np.stack([text2embedding[text] for text in val_texts])
test_embeddings = np.stack([text2embedding[text] for text in test_texts])

Loaded model nomic-ai/modernbert-embed-base to mps


Processing chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Chunk 0:   0%|          | 0/63 [00:00<?, ?it/s]

Saved 8043 embeddings to /Users/adityadawar/Documents/research/HypotheSAEs/emb_cache/liar_quickstart_nomic-ai/modernbert-embed-base/chunk_000.npy


In [23]:
train_embeddings

array([[ 0.03691504, -0.05063944,  0.0389139 , ..., -0.05135543,
         0.00529265, -0.03348878],
       [ 0.05683989, -0.00341501, -0.03449258, ..., -0.02913508,
        -0.02475017,  0.00986823],
       [ 0.04823336, -0.02630228, -0.01262546, ...,  0.01877052,
        -0.00125598, -0.02350313],
       ...,
       [ 0.03123998,  0.00484515, -0.00139259, ...,  0.02852917,
        -0.00157056, -0.030263  ],
       [ 0.06022324,  0.01731691,  0.01920618, ..., -0.02745246,
        -0.01117144,  0.01356598],
       [ 0.01089353, -0.0282669 ,  0.03712013, ...,  0.02240189,
        -0.01064814,  0.02889527]], shape=(6472, 768), dtype=float32)

In [24]:
checkpoint_dir = os.path.join("checkpoints", CACHE_NAME)
sae = train_sae(embeddings=train_embeddings, val_embeddings=val_embeddings,
                M=256, K=8, matryoshka_prefix_lengths=[32, 256], 
                checkpoint_dir=checkpoint_dir)

  0%|          | 0/100 [00:00<?, ?it/s]

Saved model to checkpoints/liar_quickstart_nomic-ai/modernbert-embed-base/SAE_matryoshka_M=256_K=8_prefixes=32-256.pt


In [25]:
model = LogisticRegression(random_state=23, C=0.3).fit(train_embeddings, labels)
train_results = model.predict(train_embeddings)
val_results = model.predict(val_embeddings)
test_results = model.predict(test_embeddings)

In [26]:
train_results

array([0, 1, 0, ..., 0, 0, 0], shape=(6472,))

In [27]:
def calculate_metrics(results, labels):
    tps = np.count_nonzero((results == labels) & (results == 1))
    fps = np.count_nonzero((results != labels) & (results == 1))
    fns = np.count_nonzero((results != labels) & (results == 0))
    accuracy = np.mean(results == labels) * 100
    precision = tps / (tps + fps)
    recall = tps / (tps + fns)
    f1 = 2 * precision*recall / (precision + recall)
    return "\n".join([format(accuracy, "Accuracy"), format(precision, "Precision"), format(recall, "Recall"), format(f1, "F1 Score")])

def format(num, label, rounding=2):
    return f"{label}: {round(num, rounding)}"

In [28]:
print("Train Results \n---------------")
print(calculate_metrics(train_results, labels))
print("\nValidation Results \n---------------")
print(calculate_metrics(val_results, val_labels))
print("\nTest Results \n---------------")
print(calculate_metrics(test_results, test_labels))

Train Results 
---------------
Accuracy: 67.2
Precision: 0.68
Recall: 0.8
F1 Score: 0.73

Validation Results 
---------------
Accuracy: 69.34
Precision: 0.67
Recall: 0.83
F1 Score: 0.74

Test Results 
---------------
Accuracy: 66.33
Precision: 0.67
Recall: 0.82
F1 Score: 0.73


In [29]:
mlp = MLPClassifier(solver='adam', alpha=1e-5, hidden_layer_sizes=(16,), early_stopping=True, random_state=23).fit(train_embeddings, labels)

In [30]:
train_results_mlp = mlp.predict(train_embeddings)
val_results_mlp = mlp.predict(val_embeddings)
test_results_mlp = mlp.predict(test_embeddings)

In [31]:
print("Train Results \n---------------")
print(calculate_metrics(train_results_mlp, labels))
print("\nValidation Results \n---------------")
print(calculate_metrics(val_results_mlp, val_labels))
print("\nTest Results \n---------------")
print(calculate_metrics(test_results_mlp, test_labels))

Train Results 
---------------
Accuracy: 69.21
Precision: 0.71
Recall: 0.77
F1 Score: 0.74

Validation Results 
---------------
Accuracy: 70.34
Precision: 0.7
Recall: 0.77
F1 Score: 0.73

Test Results 
---------------
Accuracy: 67.47
Precision: 0.69
Recall: 0.76
F1 Score: 0.73


In [33]:
# This instruction will be included in the neuron interpretation prompt.
# The below instructions are specific to Yelp, but you can customize this for your task.
# If you don't pass in task-specific instructions, there is a generic instruction (see src/interpret_neurons.py);
# task-specific instructions are optional, but they help produce hypotheses at the desired level of specificity.

TASK_SPECIFIC_INSTRUCTIONS = """All of the texts are political statements that were fact-checked.
Features should describe a specific aspect of the statement. For example:
- "uses hyperbolic language"
- "feeds into conspiracy theories'\""""

# Interpret random neurons
results = interpret_sae(
    texts=texts,
    embeddings=train_embeddings,
    sae=sae,
    n_random_neurons=5,
    print_examples_n=3,
    task_specific_instructions=TASK_SPECIFIC_INSTRUCTIONS,
    interpreter_model=INTERPRETER_MODEL,
)

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Activations shape: (6472, 256)


Generating interpretations:   0%|          | 0/5 [00:00<?, ?it/s]


Neuron 252 (1.6% active, from SAE M=256, K=8): accuses a named political opponent of unethical, dishonest, or criminal behavior

Top activating examples:
1. Dan Patrick changed his name from Danny Goeb to hide from his debts.
2. Dan Webster would force victims of rape and incest to bear their attackers child.
3. Dan Branch once lobbied for the AFL-CIO.
----------------------------------------------------------------------------------------------------

Neuron 79 (1.6% active, from SAE M=256, K=8): makes claims about hypothetical consequences or actions that would occur if a proposed policy or event happens

Top activating examples:
1. Nazi imagery [was used] by the Block for Governor campaign to describe supporters of Allan Fung for Governor.
2. Gov. Lawton Chiles said "if I were to become ... become speaker of the House it would be (his) worst nightmare."
3. Barack Hussein Obama will ... force local authorities to allow Occupy protesters to live in parks.
----------------------------

In [40]:
selection_method = "correlation"
results = generate_hypotheses(
    texts=texts,
    labels=labels,
    embeddings=train_embeddings,
    sae=sae,
    cache_name=CACHE_NAME,
    selection_method=selection_method,
    n_selected_neurons=20,
    n_candidate_interpretations=1,
    task_specific_instructions=TASK_SPECIFIC_INSTRUCTIONS,
    interpreter_model=INTERPRETER_MODEL,
    annotator_model=ANNOTATOR_MODEL,
    n_workers_annotation=N_WORKERS_ANNOTATION, # Please lower this parameter if you are running into OpenAI API rate limits
)

print("\nMost predictive features:")
pd.set_option('display.max_colwidth', None)
display(results.sort_values(by=f"target_{selection_method}", ascending=False).round(3))
pd.reset_option('display.max_colwidth')

Embeddings shape: (6472, 768)


Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Activations shape: (6472, 256)

Step 1: Selecting top 20 predictive neurons

Step 2: Interpreting selected neurons


Generating interpretations:   0%|          | 0/20 [00:00<?, ?it/s]


Step 3: Scoring Interpretations
Found 200 cached items; annotating 1800 uncached items


Scoring neuron interpretation fidelity (20 neurons; 1 candidate interps per neuron; 100 examples to score each…

API error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4.1-mini in organization org-hHdmH7ZzcQwbzYNRyUbMOgGX on requests per min (RPM): Limit 500, Used 500, Requested 1. Please try again in 120ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}; retrying in 10.0s... (2/3)
API error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4.1-mini in organization org-hHdmH7ZzcQwbzYNRyUbMOgGX on requests per min (RPM): Limit 500, Used 500, Requested 1. Please try again in 120ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}; retrying in 10.0s... (2/3)
API error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4.1-mini in organization org-hHdmH7ZzcQwbzYNRyUbMOgGX on requests per min (RPM): Limit 500, Used 500, Requested 1. Please try again in 120ms. Visit

KeyboardInterrupt: 

API error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4.1-mini in organization org-hHdmH7ZzcQwbzYNRyUbMOgGX on requests per min (RPM): Limit 500, Used 500, Requested 1. Please try again in 120ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}; retrying in 10.0s... (2/3)
API error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4.1-mini in organization org-hHdmH7ZzcQwbzYNRyUbMOgGX on tokens per min (TPM): Limit 200000, Used 200000, Requested 562. Please try again in 168ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}; retrying in 10.0s... (2/3)
API error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4.1-mini in organization org-hHdmH7ZzcQwbzYNRyUbMOgGX on tokens per min (TPM): Limit 200000, Used 200000, Requested 574. Please try again in 17

In [39]:
metrics, evaluation_df = evaluate_hypotheses(
    hypotheses_df=results,
    texts=test_texts,
    labels=test_labels,
    cache_name=CACHE_NAME,
    annotator_model=ANNOTATOR_MODEL,
    n_workers_annotation=N_WORKERS_ANNOTATION, # Please lower this parameter if you are running into OpenAI API rate limits
)

pd.set_option('display.max_colwidth', None)
display(evaluation_df.sort_values(by="separation_score", ascending=False).round(3))
pd.reset_option('display.max_colwidth')

print("\nHoldout Set Metrics:")
print(f"R² Score: {metrics['r2']:.3f}")
print(f"Significant hypotheses: {metrics['Significant'][0]}/{metrics['Significant'][1]} " 
      f"(p < {metrics['Significant'][2]:.3e})")

Step 1: Annotating texts with 20 hypotheses
Found 0 cached items; annotating 15800 uncached items


Annotating:   0%|          | 0/15800 [00:00<?, ?it/s]

API error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4.1-mini in organization org-hHdmH7ZzcQwbzYNRyUbMOgGX on requests per min (RPM): Limit 500, Used 500, Requested 1. Please try again in 120ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}; retrying in 10.0s... (2/3)
API error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4.1-mini in organization org-hHdmH7ZzcQwbzYNRyUbMOgGX on requests per min (RPM): Limit 500, Used 500, Requested 1. Please try again in 120ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}; retrying in 10.0s... (2/3)
API error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4.1-mini in organization org-hHdmH7ZzcQwbzYNRyUbMOgGX on tokens per min (TPM): Limit 200000, Used 200000, Requested 556. Please try again in 166ms.

KeyboardInterrupt: 

API error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4.1-mini in organization org-hHdmH7ZzcQwbzYNRyUbMOgGX on tokens per min (TPM): Limit 200000, Used 199589, Requested 571. Please try again in 48ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}; retrying in 10.0s... (2/3)
API error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4.1-mini in organization org-hHdmH7ZzcQwbzYNRyUbMOgGX on requests per min (RPM): Limit 500, Used 500, Requested 1. Please try again in 120ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}; retrying in 10.0s... (2/3)
API error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4.1-mini in organization org-hHdmH7ZzcQwbzYNRyUbMOgGX on requests per min (RPM): Limit 500, Used 500, Requested 1. Please try again in 120ms. Vi